## Generative Adversarial Networks
The primary objective of **Generative Adversarial Network (GAN)** is to create images that resemble (but are not identical to) those in the training dataset. Including 2 neural networks that are trained in opposition to each other:
- **Generator** takes a random vector and is tasked with producing an image from it
- **Discriminator** is a network designed to differentiate between an *original image* and one created by the **generator**. 

In [ ]:
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
generator = Sequential([
    Dense(256, input_shape=(100,)),
    LeakyReLU(alpha=0.2),
    Dense(512),
    LeakyReLU(alpha=0.2),
    BatchNormalization(momentum=0.8),
    Dense(1024),
    LeakyReLU(alpha=0.2),
    BatchNormalization(momentum=0.8),
    Dense(784, activation='tanh'),
    Reshape((28, 28))
])

optimizer = keras.optimizers.Adam(lr = 0.0002, decay = 8e-9)
generator.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

In [ ]:
discriminator = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(784),
    LeakyReLU(alpha=0.2),
    Dense(784 // 2),
    LeakyReLU(alpha=0.2),
    Dense(1, activation='sigmoid')  
])

discriminator.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

In [ ]:
discriminator.trainable = False
adversarial_model = Sequential([generator, discriminator])
adversarial_model.compile(loss='binary_crossentropy', optimizer=optimizer)

### Load Dataset

In [ ]:
(X_train,_), (_, _) = keras.datasets.mnist.load_data()
X_train = (X_train.astype(np.float32) - 127.5) / 127.5

### Network training

In [ ]:
def plotn(n, generator, device):
    noise = np.random.normal(0, 1, (n, 100))
    generated_images = generator.predict(noise)
    fig, ax = plt.subplots(1, n)
    for i in range(n):
        ax[i].imshow(generated_images.reshape(28, 28))
        
    plt.show()

In [ ]:
batch = 32

for cnt in range(3000):
    #-----Training the discriminator-----#
    idx = np.random.randint(0, len(X_train) - batch // 2)
    real_imgs = X_train[idx : idx + batch // 2].reshape(batch // 2, 28, 28)
    
    noise = np.random.normal(0, 1, (batch // 2, 100))
    fake_imgs = generator.predict(noise)
    
    x_combined_batch = np.concatenate((real_imgs, fake_imgs))
    y_combined_batch = np.concatenate((np.ones((batch // 2, 1)), np.zeros((batch // 2, 1))))
    
    d_loss = discriminator.train_on_batch(x_combined_batch, y_combined_batch)
    
    #-----Training the generator-----#
    noise = np.random.normal(0, 1, (batch, 100))
    g_loss = adversarial_model.train_on_batch(noise, np.ones((batch, 1)))
    
    if cnt % 500 == 0:
        print("Epoch: %d, Iteration: %d, D loss: %f, G loss: %f" % (cnt // 500, cnt, d_loss[0], g_loss))
        plotn(5, generator, device='cpu')

## Deep Convolutional GANs (DCGANs)
This model uses convolutional layers for both the generator and the discriminator. The key distinction here is the use of `Conv2DTranspose` layer in the **generator**. 

### Load data

In [ ]:
(X_train, _), (_, _) = keras.datasets.mnist.load_data()
X_train = (X_train.astype(np.float32)-127.5) / 127.5
print(X_train.min(),X_train.max())

In [ ]:
generator = Sequential([
    Dense(128 * 7 * 7, activation="relu", input_dim = 100),
    Reshape((7, 7, 128)),
    UpSampling2D(),
    Conv2DTranspose(128, kernel_size=3, padding="same"),
    BatchNormalization(momentum=0.8),
    Activation("relu"),
    UpSampling2D(),
    Conv2DTranspose(64, kernel_size=3, padding="same"),
    BatchNormalization(momentum=0.8),
    Activation("relu"),
    UpSampling2D(),
    Conv2DTranspose(1, kernel_size=3, padding="same"),
    Activation("tanh")
])

optimizer = keras.optimizers.Adam(lr = 0.0001)
generator.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
generator.summary()

In [ ]:
discriminator = Sequential([
    Conv2D(32, kernel_size = 3, strides=2, input_shape=(28, 28, 1), padding="same"),
    LeakyRelu(alpha=0.2),
    Dropout(0.25),
    Conv2D(64, kernel_size = 3, strides=2, padding="same"),
    ZeroPadding2D(padding=((0,1),(0,1))),
    BatchNormalization(momentum=0.8),
    LeakyRelu(alpha=0.2),
    Dropout(0.25),
    Conv2D(128, kernel_size = 3, strides=2, padding="same"),
    BatchNormalization(momentum=0.8),
    LeakyRelu(alpha=0.2),
    Dropout(0.25),
    Conv2D(128, kernel_size = 3, strides=1, padding="same"),
    BatchNormalization(momentum=0.8),
    LeakyRelu(alpha=0.2),
    Dropout(0.25),
    Conv2D(256, kernel_size = 3, strides=1, padding="same"),
    BatchNormalization(momentum=0.8),
    LeakyRelu(alpha=0.2),
    Dropout(0.25),
    Flatten(),
    Dense(1, activation='sigmoid')
])

discriminator.compile(loss='binary_crossentropy', optimizer=optimizer)

In [ ]:
discriminator.trainable = False

adversarial_model = Sequential([generator, discriminator])
adversarial_model.compile(loss='binary_crossentropy', optimizer=optimizer)

In [ ]:
batch = 32
y_labeled = np.ones((batch, 1))
y_mislabeled = np.zeros((batch, 1))
for cnt in range(1000):
    #-----Training the discriminator-----#
    idx = np.random.randint(0, len(X_train) - batch // 2)
    real_imgs = X_train[idx : idx + batch // 2].reshape(batch // 2, 28, 28, 1)

    noise = np.random.normal(0, 1, (batch, 100))
    fake_imgs = generator.predict(noise)
    
    d_loss_1 = discriminator.train_on_batch(real_imgs, y_labeled)
    d_loss_2 = discriminator.train_on_batch(fake_imgs, y_mislabeled)

    d_loss = 0.5 * np.add(d_loss_1, d_loss_2)

    #-----Training the generator-----#
    g_loss = adversarial_model.train_on_batch(noise, y_labeled)

    if cnt % 100 == 0:
        print("Epoch: %d, Iteration: %d, D loss: %f, G loss: %f" % (cnt // 500, cnt, d_loss[0], g_loss))
        plotn(5, generator, device='cpu')